## Evaluation of Generated Clinical Notes using BERTScore and BERT Embeddings

This notebook evaluates the generated clinical notes by comparing them to the reference notes using the BERTScore evaluation metric. The BERTScore uses cosine similarity between BERT embeddings to calculate precision, recall, and F1 scores for each sentence. 

The results from this evaluation will provide insights into the quality of the generated clinical notes based on semantic similarity to the reference notes.

<hr style="height:1rem;">

## Imports

In [1]:
import os
import re
import time

from pathlib import Path
import pandas as pd

from bert_score import score
from transformers import BertModel, BertTokenizer
from transformers import BertForPreTraining

# pytorch library
import torch # the main pytorch library
import torch.nn.functional as f # the sub-library containing different functions for manipulating with tensors

import nltk
from nltk.tokenize import sent_tokenize

In [2]:
import warnings
warnings.filterwarnings("ignore")

import logging
from transformers import logging as transformers_logging

# Suppress specific logs from transformers
transformers_logging.set_verbosity_error()

In [3]:
# Download necessary NLTK data
nltk.download('punkt')

[nltk_data] Downloading package punkt to D:\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

<hr style="height:1rem;">

## Path Settings

In [4]:
NOTEBOOK_PATH = os.getcwd()
NOTEBOOK_DATA_PATH = os.path.join(
    NOTEBOOK_PATH,
    'data'
)
NOTEBOOK_EVALUATION_PATH = os.path.join(
    NOTEBOOK_PATH,
    'evaluation'
)
NOTEBOOK_RESULTS_PATH = os.path.join(
    NOTEBOOK_PATH,
    'results'
)
NOTEBOOK_PLOTS_PATH = os.path.join(
    NOTEBOOK_PATH,
    'plots'
)
DISSERTATION_PATH = Path.cwd().resolve().parents[0]
DISSERTATION_DATA_PATH = os.path.join(
    DISSERTATION_PATH,
    'data'
)

print(NOTEBOOK_PATH)
print(NOTEBOOK_DATA_PATH)
print(NOTEBOOK_RESULTS_PATH)
print(NOTEBOOK_EVALUATION_PATH)
print(NOTEBOOK_PLOTS_PATH)
print(DISSERTATION_PATH)
print(DISSERTATION_DATA_PATH)

D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\data
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\results
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\evaluation
D:\ODU\03 - Dissertation Defense\COT-KG\latest\samples\plots
D:\ODU\03 - Dissertation Defense\COT-KG\latest
D:\ODU\03 - Dissertation Defense\COT-KG\latest\data


<hr style="height:1rem;">

## Device Setting

With pytorch we are able to move the python calculations to the GPU. To do this we define the device on which we wish to run the calculations. Depending if cuda (the GPU drivers that enable running calculations on the graphic card) is enabled on the machine, we define the device as follows:

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


<hr style="height:1rem;">

## Variable Declarations

In [6]:
ROBERTA_BASE = "roberta-base"
MICROSOFT_DEBERTA_XL_MNLI = "microsoft/deberta-xlarge-mnli"
BERT_UNCASED = "bert-base-uncased"
BERT_LARGE_UNCASED = "bert-large-uncased"

# BioBERT, SciBERT, and ClinicalBERT
BIOBERT_BASED_CASED_V1_1 = "dmis-lab/biobert-base-cased-v1.1"
BIOBERT_BASED_CASED_V1_2 = "dmis-lab/biobert-base-cased-v1.2"
SCIBERT_SCIVOCAB_UNCASED = "allenai/scibert_scivocab_uncased"
CLINICAL_BERT = "emilyalsentzer/Bio_ClinicalBERT"

MODEL_TYPE = BERT_LARGE_UNCASED
print(MODEL_TYPE)

bert-large-uncased


<hr style="height:1rem;">

## Function Declarations

In [7]:
def parse_sentences(text):
    """
    Tokenize the input text into individual sentences.

    This function uses NLTK's sentence tokenizer to segment the input text into sentences.

    Parameters:
    - text (str): The input text to be tokenized into sentences.

    Returns:
    - List[str]: A list of sentences from the input text.
    """
    # Use NLTK's sentence tokenizer to segment the input text into sentences
    return sent_tokenize(text)

In [8]:
def align_sentences(ground_truth_sentences, generated_sentences):
    """
    Align sentences from ground truth and generated outputs using a simple positional strategy.

    This method truncates both lists to the length of the shorter one, 
    assuming that sentence order and count roughly match.

    Parameters:
    - ground_truth_sentences (List[str]): List of reference sentences.
    - generated_sentences (List[str]): List of machine-generated sentences.

    Returns:
    - Tuple (List[str], List[str]): Aligned ground truth and generated sentences of equal length.
    """
    
    # Determine the length of the shorter list to avoid index errors
    min_len = min(len(ground_truth_sentences), len(generated_sentences))

    # Truncate both lists to the same length
    aligned_ground_truth = ground_truth_sentences[:min_len]
    aligned_generated = generated_sentences[:min_len]
    
    # Return the aligned sentence lists
    return aligned_ground_truth, aligned_generated    

In [9]:
def get_bertscore(ground_truth, machine_generated):
    """
    Compute the average BERTScore (Precision, Recall, F1) between ground truth and machine-generated text.

    Parameters:
    - ground_truth (str): The reference text written by a human.
    - machine_generated (str): The text generated by a machine (e.g., an LLM).

    Returns:
    - Tuple (float, float, float): Mean Precision, Recall, and F1 BERTScore across aligned sentence pairs.
    """
    
    # Split the ground truth text into individual sentences
    ground_truth_sentences = parse_sentences(ground_truth)

    # Split the machine-generated text into individual sentences
    generated_sentences = parse_sentences(machine_generated)
    
    # Align the ground truth and generated sentences for one-to-one comparison
    aligned_ground_truth, aligned_generated = align_sentences(ground_truth_sentences, generated_sentences)
    
    # Compute BERTScore precision (P), recall (R), and F1 score (F1)
    P, R, F1 = score(
        aligned_generated,                           # Hypothesis sentences (model outputs)
        aligned_ground_truth,                        # Reference sentences (ground truth)
        lang = "en",                                 # Language setting for BERTScore
        #batch_size = 64,                             # Optional: control memory and speed
        #nthreads = 4,                                # Optional: use multiple CPU threads
        #num_layers = 10,                             # Optional: choose BERT layer (default is usually the last)      
        model_type = MODEL_TYPE,                     # Predefined model type for embeddings (e.g., "bert-large-cased")
        verbose = False                              # Suppress verbose logging
    )
    
    # Return the mean precision, recall, and F1 score as scalar floats
    return P.mean().item(), R.mean().item(), F1.mean().item()

In [10]:
def get_document_bertscore(ground_truth, machine_generated):
    """
    Compute the BERTScore (Precision, Recall, F1) between full ground truth and machine-generated documents.

    This function treats the entire input texts as single documents rather than sentence-aligned segments.

    Parameters:
    - ground_truth (str): The reference human-written text.
    - machine_generated (str): The machine-generated text to evaluate.

    Returns:
    - Tuple (float, float, float): Mean BERTScore Precision, Recall, and F1 across all text segments.
    """

    # Calculate BERTScore directly on the full document texts (as single-element lists)
    P, R, F1 = score(
        [machine_generated],       # the machine output as a single-element list
        [ground_truth],            # the ground truth similarly
        lang = "en",              # Language setting for the BERT model
        # batch_size = 64,        # Optional: size of mini-batches
        # nthreads = 4,           # Optional: number of threads for processing
        # num_layers = 10,        # Optional: specific BERT layer to use for embeddings
        model_type = MODEL_TYPE, # Pre-trained BERT model type (e.g., "bert-large-cased")
        verbose = False          # Disable verbose output
    )
    
    # Return the mean precision, recall, and F1 score as scalar floats
    return P.mean().item(), R.mean().item(), F1.mean().item()

In [11]:
def compute_cosine_distances(ground_truth_text, generated_text, model, tokenizer, device):
    # Due to the token size of our GPU - 512 tokens, we need to trim off (256 for ground-truth text and 256 for generated text)    
    tokens = tokenizer.tokenize(generated_text)
    if len(tokens) > 256:
        # Truncate to the maximum token length
        tokens = tokens[:256]

        # Convert the tokens back to a string
        generated_text = tokenizer.convert_tokens_to_string(tokens)         
    
    texts  = [
        ground_truth_text,
        generated_text
    ]

    # Get the encodings of the sentences. Since the sentences are of different lenghts we set the 'padding' parameter 
    # to 'True' which will pad the shorter sentences. We also set the return_tensors to 'pt' so that we get the 'pytorch' 
    # tensors instead of python lists.
    encodings = tokenizer(
        texts, # the texts to be tokenized
        padding=True, # pad the texts to the maximum length (so that all outputs have the same length)
        return_tensors='pt' # return the tensors (not lists)
    )

    # The input tensors need to be on the same device as the model. In order for us to use the BERT model to get the 
    # sentence embeddings we must first move the input tensors to the same device as the model. 
    encodings = encodings.to(device)

    # In order to get the token embeddings we will send the encodings through the model. Pytorch by default calculates 
    # the gradients for every layer of the model which is really handy when training the model but not when we want to 
    # use it in production. Gradients take additional space on CPU or GPU which you don't want to populate with things 
    # you don't need.  We can disable gradient calculation by adding the 'with torch.no_grad():' before the model.
    
    # disable gradient calculations
    with torch.no_grad():
        # get the model embeddings
        embeds = model(**encodings)    
    
    # The output of the model is a tuple of tensors:
    #   The first tensor (embeds[0]) contains the embeddings of the sentence tokens.
    #   The second tensor (embeds[1]) contains the embeddings of the '[CLS]' token that can be then sent further to 
    #     train the next sentence prediction. We won't use this tensor in this notebook.
    embeds = embeds[0]
    
    # Way 1: Use the [CLS] Embeddings
    # We will take the first embedding representing the '[CLS]' token of all three sentences and calculate the 
    # cosine distance.
    CLSs = embeds[:, 0, :]
    
    # normalize the CLS token embeddings
    normalized = f.normalize(CLSs, p=2, dim=1)
    
    # calculate the cosine similarity
    cls_dist = normalized.matmul(normalized.T).cpu()
    cls_dist = cls_dist.new_ones(cls_dist.shape) - cls_dist
    cls_dist = cls_dist.numpy()    
    
    # Way 2: Computing the Mean of All Output Vectors
    # We will calculate the mean of all embeddings for each sentence and calculate the cosine distance. 
    # To get the means we will use the '.mean' method that is documented here:
    #     https://pytorch.org/docs/stable/generated/torch.mean.html#torch.mean
    MEANS = embeds.mean(dim=1)
    
    # normalize the MEANS token embeddings
    normalized = f.normalize(MEANS, p=2, dim=1)
    
    # calculate the cosine similarity
    mean_dist = normalized.matmul(normalized.T).cpu()
    mean_dist = mean_dist.new_ones(mean_dist.shape) - mean_dist
    mean_dist = mean_dist.numpy()
    
    # Way 3: Compute the max-over-time of the Output Vectors
    # We will calculate the max-over-time of all embeddings for each sentence and calculate the cosine distance. 
    # The max-over-time is basically getting the maximum value for each dimension of the tokens. 
    # We will use the .max method that is documented here: 
    #     https://pytorch.org/docs/stable/generated/torch.max.html#torch.max
    # The .max method returns a tuple where the first value is a tensor of maximum values based on the dim argument, 
    # and the second value are the indices of the index location of each maximum value found ('argmax').
    MAXS, _ = embeds.max(dim=1)
    
    # normalize the MEANS token embeddings
    normalized = f.normalize(MAXS, p=2, dim=1)
    
    # calculate the cosine similarity
    max_dist = normalized.matmul(normalized.T).cpu()
    max_dist = max_dist.new_ones(max_dist.shape) - max_dist
    max_dist = max_dist.numpy()

    # Return the lists of distances
    return cls_dist[0][1], mean_dist[0][1], max_dist[0][1]

<hr style="height:1rem;">

## Read Data

### Read Ground-Truth Data from CodiEsp

In [12]:
codiesp_file = "../data/samples/codiesp_mondo_samples.csv"
codiesp_df = pd.read_csv(codiesp_file)
codiesp_df['icd_codes'] = codiesp_df['icd_codes'].apply(lambda x: ' '.join([code.split('.')[0] for code in x.split()]))
len(codiesp_df)
codiesp_df.head(5)

,article_id,text_references,icd_codes,age,gender,text,icd_match_count,total_icd_count,bin
0,S1575-06202010000200004-1,seborrheic dermatitis; TB brain injuries; HCV;...,L21 A17 B18 B59 B37 A15 B58 L40 Z65 B20 R64 J1...,40,male,We report the case of a 40-year-old male patie...,7,21,bin7
1,S1134-80462015000100006-1,pulmonary edema; hemodialysis; breathlessness;...,J81 5A1D00Z R06 R40 R69 L53 M54 R52 I10 E03 N2...,49,female,A 49-year-old female patient (weight: 60 kg; h...,7,23,bin7
2,S1130-05582015000100008-1,lymph nodes; right lower jaw biopsy; oral cavi...,R59 0NBT 0CB4 R68 0W95 F17 F12 B19 B20 K04 R59...,24,female,A 24-year-old woman with reactive HIV serology...,5,23,bin5
3,S0378-48352004001000003-1,abdominal aortic aneurysm; asthenia; anorexy; ...,I71 R53 R63 E80 R17 R82 K82 R16 B20 J99 B15 B1...,69,male,A 69-year-old male with a history of stage D2 ...,6,33,bin6
4,S0212-16112010000600024-1,stress; disease; weightloss; infection; sepsis...,F43 R69 R63 B99 A41 K25 C44 R10 R11 0YQ5 R10 E...,77,male,We report the case of a 77-year-old man who ca...,8,33,bin8


<hr style="height:1rem;">

## Metrics Evaluations

### METRIC #1 - Sentence Similarity using BERT

Calculate the similarity of different documents using BERT. Two documents will be sent through the BERT model to retrieve its embeddings and use them to calculate the document similarity by:

- Using the [CLS] token of each sentence.
- Calculating the mean of sentence embeddings.
- Calculating the max-over-time of the sentence embeddings.

The python libraries that we will use in this notebook are:

- transformers. The huggingface library containing the general-purpose architectures for NLP.
- torch. An open source machine learning framework used to create and train models on the GPU.
- matplotlib. A comprehensive library for creating static, animated and interactive visualizations in Python.

#### Measure the Sentence Similarities

Using the sentence token embeddings, we will calculate the sentence similarities. We will do this in three ways:

- Using the ```[CLS]``` token embeddings.
- Calculating the mean of all sentence token embeddings.
- Calculating the max-over-time of the sentence token embeddings.

In all three ways we will calculate the sentence similarities using the cosine distance.

<hr style="height:0.25rem;">

#### BERT Tokenizer and Model

Now we will import a pretrained BERT model from Hugging Face. There are a lot of different BERT models that were trained with different hyperparameters and tasks. A full list of official models is available [here](https://huggingface.co/transformers/pretrained_models.html), but there are also a lot of models that are uploaded by the [community](https://huggingface.co/models).

##### BERT Model

In this notebook we will use the ```bert-large-cased``` model. To use it we will need to import both the ```tokenizer``` and the ```model```: The ```tokenizer``` will enable us to transform strings into tensors that can be then sent to the model, which in turn will give us the embeddings.

##### BERT Tokenizer

The tokenizer transforms a string into lists, tensors, etc. which can be used as the input of the model. Different tokenizers have different inputs. The BERT tokenizer takes the following inputs:

- ```text```. The sequence or batch of sequences to be encoded.
- ```text_pair```. The OTHER sequence of batch of sequences to be encoded (used for next sentence prediction). (Default: None)
- ```padding```. Activates and controls padding. If True, pad to the longest sequence in the batch (or no padding if only a single sequence is provided). Other options are also available. (Default: False)
- ```truncation```. Activates and controls truncation. If True, Truncate to a maximum length specified with the ```max_length``` argument. (Default: False)
- ```max_length```. Controls the maximum length to use by one of the truncation/padding parameters.
return_tensors. If set, will return tensors instead of list of python integers.

The full documentation of the BERT tokenizer can be found [here](https://huggingface.co/transformers/model_doc/bert.html#berttokenizer). NOTE: The BERT tokenizer inherits from the ```PreTrainedTokenizer``` class, which is documented [here](https://huggingface.co/transformers/main_classes/tokenizer.html#transformers.PreTrainedTokenizer).


The BERT tokenizer returns three values:

- ```input_ids```. The list/tensor of token ids for each sentence. The id ```101``` represents the class token ```[CLS]```, ```102``` represents the separator token ```[SEP]```, and 0 represents the padding token ```[PAD]```.
- ```attention_mask```. This list/tensor represents which ids to use when generating the tokens (e.g. ignores the [PAD] tokens).
- ```token_type_ids```. This list/tensor represents which tokens correspond to the first and second sentence (used for next sentence prediction).

<hr style="height:0.5rem;">

### METRIC #2 - BERTScore

####  Evaluating using BERTScore
ROUGE relies on the exact presence of words in both the predicted and reference texts, failing to interpret the underlying semantics. This is where [BERTScore](https://arxiv.org/abs/1904.09675) comes in and leverages the contextual embeddings from the BERT model, aiming to evaluate the similarity between a predicted and a reference sentence in the context of machine-generated text. By comparing embeddings from both sentences, `BERTScore` captures semantic similarities that might be missed by traditional n-gram based metrics.

#### Supported Models:
https://docs.google.com/spreadsheets/d/1RKOVpselB98Nnh_EOC4A2BYn8_201tmPODpNWu4w7xI/edit?gid=0#gid=0

In [13]:
# Load the pre-trained BERT model (without any task-specific head)
model = BertModel.from_pretrained(BERT_LARGE_UNCASED)
#model = BertForPreTraining.from_pretrained(BERT_LARGE_UNCASED)

# Load the tokenizer for the specified BERT model
tokenizer = BertTokenizer.from_pretrained(BERT_LARGE_UNCASED)

# Set the model to evaluation mode (disables dropout and gradient updates)
model = model.eval()

# Move the model to the specified device (e.g., CPU or GPU)
model = model.to(device)

In [14]:
ARTICLE_IDS = [
    'S1575-06202010000200004-1', # 0
    'S1134-80462015000100006-1', # 1
    'S1130-05582015000100008-1', # 2
    'S0378-48352004001000003-1', # 3
    'S0212-16112010000600024-1', # 4
    'S0004-06142006000700014-1', # 5
    'S2254-28842013000300009-1', # 6
    'S1699-695X2016000200009-1', # 7
    'S1130-63432015000400006-1', # 8
    'S0213-12852006000600002-1', # 9
    'S0212-71992007000100006-1', # 10
    'S0212-16112005000600011-1', # 11
    'S0211-69952011000500013-1', # 12
    'S0210-56912009000900008-1', # 13
    'S0004-06142009000900012-1', # 14
    'S1699-695X2015000100011-1', # 15
    'S1130-63432015000200008-1', # 16
    'S1130-01082009000400015-1', # 17
    'S0378-48352006000300005-1', # 18
    'S0365-66912007000100010-1', # 19
    'S0365-66912004001100011-1', # 20
    'S0212-71992004001100007-1', # 21
    'S0212-71992004000300009-1'  # 22
]

In [15]:
match_article_id = ARTICLE_IDS[10]
print(match_article_id)

S0212-71992007000100006-1


##### CoT KG Samples

In [16]:
COT_KG_GENERATED_TEXTS = [
    """A 52-year-old male reports experiencing recurring headaches, intermittent facial weakness, and abnormal sensations in his extremities that have persisted for several weeks. He describes the facial discomfort as episodic and accompanied by a feeling of numbness that occasionally affects his daily activities. He also notes a longstanding history of an inherited hemoglobinopathy which he believes may be contributing to his overall fatigue and malaise. The patient mentions his ongoing nicotine use and previous viral infections, which he suspects might be influencing his current neurological symptoms. He denies any acute changes or recent trauma and is seeking evaluation for better symptom management.""",
    """A 52-year-old male reports episodes of headache and intermittent facial weakness accompanied by numbness and coordination difficulties that have been present for several weeks. He describes the facial discomfort as moderately severe, with periods of altered sensation that occasionally disrupt his daily activities. He also mentions persistent fatigue, which he attributes in part to his known history of thalassemia. Additionally, he notes a longstanding history of tobacco use and previous episodes of systemic symptoms suggestive of infectious exposure. He is concerned that these overlapping neurological and systemic symptoms may be interrelated and is seeking further evaluation and management.""",
    """A 52-year-old male reports the sudden onset of unilateral facial weakness accompanied by difficulty with speech and a persistent, moderate headache over the past few days. He also describes intermittent numbness and tingling in his extremities with episodes of unsteadiness and overall fatigue that have gradually worsened. The patient notes occasional dental discomfort and has a background of thalassemia, which he acknowledges as a chronic condition. He has a history of nicotine and alcohol use, which he feels may be contributing to his overall sense of malaise. He is seeking evaluation to better understand these neurologic and systemic symptoms and determine their underlying cause.""",
    
    """A 52-year-old male with known HIV infection, thalassemia trait, and history of Lyme disease and treated syphilis reports a five-day history of progressive right-sided facial weakness and perioral numbness. He describes associated dysarthria, severe occipital headache, and intermittent bilateral hand paresthesias with unsteadiness on standing. He has experienced low-grade fevers, night sweats, and profound fatigue for the past two weeks, and notes a resolving herpetic lip eruption last week. He admits to daily tobacco and alcohol use and denies recent dental procedures despite persistent mild tooth pain. No bowel or bladder changes are reported.""",
    """A 52-year-old male reports a 2-week history of progressive right-sided facial weakness with intermittent facial twitching, accompanied by numbness and tingling in his hands and feet and mild gait unsteadiness. He describes daily throbbing headaches rated 6/10 and marked fatigue limiting his usual activities. Over the past month he has experienced sharp, spontaneous dental pain triggered by hot or cold and occasional swelling under his right jaw consistent with salivary gland discomfort. He also notes grouped vesicular lesions on his lips and lower face that periodically crust over. He admits to long-standing heavy tobacco use, occasional alcohol intake, and a known penicillin allergy.""",
    """A 52-year-old male with known HIV infection and nicotine dependence reports a two-week history of progressive left-sided facial weakness and numbness accompanied by intermittent shooting pain. He notes mild slurred speech with transient word-finding difficulty over the past several days. He also endorses persistent low-grade headache, malaise, and an unsteady gait with occasional falls. He denies fever, recent dental work, or trauma but recalls a tick bite while hiking three weeks ago. His symptoms are moderate in severity and have steadily worsened despite over-the-counter analgesics.""",
    """A 52-year-old male reports a 3-week history of intermittent, throbbing headaches with perioral numbness and right-sided facial weakness, described as drooping of the eyelid and occasional slurred speech. He has experienced transient episodes of word-finding difficulty and dysarthria, accompanied by an unsteady gait and bilateral lower extremity tingling and numbness after prolonged standing. He endorses low-grade fevers, night sweats, malaise, and diffuse myalgias following regular consumption of unpasteurized dairy on his farm. He also notes chronic left mandibular tooth pain with episodic swelling and sensitivity consistent with pulpal inflammation. His medical history is notable for beta-thalassemia trait, nicotine dependence, and a penicillin allergy.""",
    """A 52-year-old male with beta-thalassemia and nicotine dependence reports a 5-day history of progressive right-sided facial droop and ear pain, accompanied by perioral numbness and decreased lacrimation. He endorses intermittent holocranial headaches, night sweats, and profound fatigue. Over the past week he has experienced sharp throbbing pain in the right lower molar region with heat- and cold-sensitivity and localized swelling. He also notes episodic unsteady gait and difficulty with fine motor coordination. He has a known penicillin allergy and denies rash, visual disturbances, or bowel/bladder changes.""",
    """A 52-year-old male reports a 3-week history of progressive left-sided facial paralysis with intermittent hemifacial spasms, hyperacusis and decreased taste sensation. Ten days ago he developed an erythematous, vesicular rash on his left eyelid followed by deep ear pain and worsening facial weakness. Over the same period he has experienced intermittent headaches, generalized malaise and new right lower-extremity numbness causing unsteadiness when walking. He denies fever, chest pain or bowel/bladder dysfunction. He has a history of HIV infection and treated syphilis and continues to smoke one pack per day with daily alcohol use.""",
    """A 52-year-old male reports a 5-day history of burning pain and grouped vesicular rash in the left preauricular and ear canal region, followed 3 days later by progressive left facial weakness with difficulty closing the eye and hyperacusis. He describes the pain as sharp, 6/10 in intensity, partially relieved by over-the-counter analgesics, and notes associated headache and mild gait unsteadiness without falls. He denies fever, visual changes, or dysphagia but admits to intermittent tinnitus and decreased tearing on the affected side. He has a 30-pack-year smoking history, occasional alcohol use, and a remote history of treated Lyme disease.""",
    """A 52-year-old male reports a five-day history of right-sided facial weakness with intermittent perioral twitching and altered taste sensation, accompanied by a mild frontal headache. He describes episodic tingling and numbness across the right cheek and jaw, with thermal sensitivity of his lower teeth over the past two weeks. Additionally, he notes progressive gait unsteadiness and lower extremity numbness for approximately one month, leading to occasional imbalance. He endorses generalized fatigue and malaise since the onset of his neurological symptoms. His medical history is notable for mild thalassemia trait, nicotine dependence, and previous episodes of dental pulp inflammation.""",
    """A 52-year-old male with well-controlled HIV on antiretroviral therapy and a remote history of Lyme disease reports a 5-day history of progressive right-sided facial weakness marked by twitching and incomplete eyelid closure. He describes associated periorbital tingling, persistent tearing with right lacrimal sac tenderness, and a dull headache, accompanied by generalized fatigue worsened by minimal activity. Over the past 48 hours he has noticed mild gait unsteadiness and intermittent paresthesias in his right lower extremity. He denies fever, rash, recent dental pain, or changes in his medication regimen. He is a current smoker with occasional alcohol use.""",
    """A 52-year-old male reports a three-week history of progressive right-sided facial muscle weakness with inability to fully close the eyelid and mild dysarthria, preceded by a week of malaise and throbbing frontal headaches. He describes intermittent numbness and tingling of the face and hands and has noted an unsteady gait with occasional falls over the past two weeks. He endorses persistent low-grade fevers, night sweats, generalized malaise and fatigue for one month, along with intermittent arthralgias. His background includes thalassemia trait with baseline mild anemia, active one-pack-per-day cigarette use, social alcohol consumption, and a documented penicillin allergy. He also reports episodic sharp tooth pain and sensitivity suggestive of pulpal inflammation but denies chest pain, dyspnea, or abdominal complaints."""
]


##### CoT ICD Code KG Samples

In [17]:
COT_ICD_CODE_KG_GENERATED_TEXTS = [
    """A 52-year-old male with a complex past medical history, including thalassemia, chronic nicotine and alcohol use, and immunocompromise secondary to HIV infection, presents with persistent headache and recent onset of facial weakness accompanied by dysarthria. He describes intermittent numbness and tingling in his extremities as well as difficulty with speech, which have gradually worsened over the past week. His history is notable for prior viral infections, including toxoplasmosis, rubella, measles, herpes simplex, varicella, and cytomegalovirus, and he has experienced recurrent dental infections with associated oral discomfort. He also reports allergies to certain medications, which complicate his treatment options. On examination, there is evidence of unilateral peripheral facial paresis with altered sensation along the facial distribution, in addition to subtle gait instability and motor deficits suggestive of earlier cerebrovascular events. This constellation of infectious exposures in the context of his immunosuppressed state raises concern for opportunistic neurologic complications and warrants further evaluation with neuroimaging and laboratory studies to guide management.""",
    """A 52-year-old male presents with recurrent headaches, facial discomfort, and intermittent episodes of gait instability and numbness. He reports generalized fatigue and episodes of weakness that have persisted over several weeks. He endorses a history of prior viral illnesses, including infections consistent with rubella, herpes, and toxoplasmosis, and notes chronic nicotine and alcohol use. Additionally, he is aware of an inherited hemoglobinopathy that may be contributing to his chronic symptoms.""",
    """A 52-year-old male reports the recent onset of headache accompanied by episodes of facial asymmetry and difficulty with speech. He describes a gradual worsening of unsteadiness and occasional coordination issues, which he finds particularly concerning. The patient also notes persistent fatigue and intermittent visual disturbances over the past several days. He mentions a known history of an inherited hemoglobinopathy and significant past infectious exposures, as well as long-standing nicotine dependence. Overall, he is seeking evaluation for these neurologic and systemic symptoms that have developed over a brief period.""",
    """A 52-year-old male reports several weeks of intermittent headache accompanied by episodes of altered speech and difficulty with facial movements. He describes sensations of numbness and abnormal tingling in his extremities and notes that these neurological changes have been episodic in nature. He also mentions enduring fatigue and a pervasive sense of malaise that seem to coincide with periods of mild fever and systemic discomfort. In addition, he has a longstanding history of nicotine dependence and a known blood disorder consistent with thalassemia, which he feels may be contributing to his overall fatigue. The patient is concerned about these recurring symptoms and is seeking further evaluation to clarify their underlying cause.""",
]

In [18]:
generated_text = COT_KG_GENERATED_TEXTS[5]
#generated_text = COT_ICD_CODE_KG_GENERATED_TEXTS

print(generated_text)

A 52-year-old male with known HIV infection and nicotine dependence reports a two-week history of progressive left-sided facial weakness and numbness accompanied by intermittent shooting pain. He notes mild slurred speech with transient word-finding difficulty over the past several days. He also endorses persistent low-grade headache, malaise, and an unsteady gait with occasional falls. He denies fever, recent dental work, or trauma but recalls a tick bite while hiking three weeks ago. His symptoms are moderate in severity and have steadily worsened despite over-the-counter analgesics.


<hr style="height:0.5rem;">

### Evaluate the Results


In [19]:
# Loop through a predefined number of runs
for index, row in codiesp_df.iterrows():
    article_id=str(row['article_id']) # used for title and filename

    if match_article_id != article_id:
        #print(f"Skipping (in IGNORE list): {article_id}")
        continue

    orig_generated_text = generated_text
    orig_ground_truth_text=str(row['text']) # used for title and filename
    print('\narticle_id:', article_id)
        
    # Due to the token size of our GPU - 512 tokens, we need to trim off (256 for ground-truth text and 256 for generated text)    
    ground_truth_tokens = tokenizer.tokenize(orig_ground_truth_text)
    if len(ground_truth_tokens) > 256:
        # Truncate to the maximum token length
        ground_truth_tokens = ground_truth_tokens[:256]

        # Convert the tokens back to a string
        ground_truth_text = tokenizer.convert_tokens_to_string(ground_truth_tokens) 
    else:
        ground_truth_text = orig_ground_truth_text 
    print('\nground_truth_text:', ground_truth_text)

    generated_text_tokens = tokenizer.tokenize(orig_generated_text)
    if len(generated_text_tokens) > 256:
        # Truncate to the maximum token length
        generated_text_tokens = generated_text_tokens[:256]

        # Convert the tokens back to a string
        generated_text = tokenizer.convert_tokens_to_string(generated_text_tokens) 
    else:
        generated_text = orig_generated_text 
    print('\ngenerated_text:', generated_text)

    # Metric #1 Sentence Similarity using BERT
    print('\nMetric #1')
    cls_dist, mean_dist, max_dist = compute_cosine_distances(
        ground_truth_text, generated_text, model, tokenizer, device
    )
    print('cls_dist=', cls_dist)
    print('mean_dist=', mean_dist)
    print('max_dist=', max_dist)
    
    # Metric #2 BERTScore (sentence-level)
    print('\nMetric #2')
    P_sent, R_sent, F1_sent = get_bertscore(
        ground_truth_text, generated_text
    )
    print('P_sent=', P_sent)
    print('R_sent=', R_sent)
    print('F1_sent=', F1_sent)
    
    # Metric #2 BERTScore (document-level)
    print('\nMetric #3')
    P_doc, R_doc, F1_doc = get_document_bertscore(
        ground_truth_text, generated_text
    )
    print('P_doc=', P_doc)
    print('R_doc=', R_doc)
    print('F1_doc=', F1_doc)


article_id: S0212-71992007000100006-1

ground_truth_text: a 52 - year - old man with a history of hysterectomy at age 8 due to possible b - thalassemia and no cardiovascular risk factors . smoker of 1 pack / day and moderate drinker . allergic to pyrazolones . risky sexual relations in youth he was admitted for presenting with a chronic dental infection , headache with oral discomfort for a week that had been added 48 hours previously left facial paralysis , sudden onset , bilateral without other symptoms . on physical examination , the patient presented angioma in the forehead and right eye , together with bilateral parotid swelling . the initial neurological examination showed bilateral peripheral facial paralysis that caused dysarthria with bilateral bell ' s sign . lacrimation and hearing were not affected . taste disturbance in the anterior third of the tongue . during the days after admission , paresthesias in the fingers of the hands were added , initially in the upper right li